In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
SUBURB_FILE = next(RAW.glob("Moving annual*.xlsx"))
LGA_FILE    = next(RAW.glob("quarterly-median*.xlsx"))

xl = pd.ExcelFile(SUBURB_FILE)
print(xl.sheet_names)

['1 bedroom flat', '2 bedroom flat', '3 bedroom flat', '2 bedroom house', '3 bedroom house', '4 bedroom house', 'All properties']


In [2]:
import sys; sys.path.append("../scripts")
from dffh import (load_dffh_suburb_panel, coverage_filter, growth_cagr,
                  area_to_suburbs, make_suburb_key, AREA_ALIASES)

panel = load_dffh_suburb_panel(SUBURB_FILE)

print(panel.shape)
print(panel.head())
print(f"areas: {panel.area.nunique()} | types: {panel.property_type.nunique()}")
print(f"quarters: {panel.quarter.min()} -> {panel.quarter.max()}")
print(f"missing medians: {panel.median_rent.isna().mean():.1%}")

(105266, 6)
            region                                   area   property_type  \
0  Inner Melbourne  Albert Park-Middle Park-West St Kilda  1 bedroom flat   
1  Inner Melbourne  Albert Park-Middle Park-West St Kilda  1 bedroom flat   
2  Inner Melbourne  Albert Park-Middle Park-West St Kilda  1 bedroom flat   
3  Inner Melbourne  Albert Park-Middle Park-West St Kilda  1 bedroom flat   
4  Inner Melbourne  Albert Park-Middle Park-West St Kilda  1 bedroom flat   

  quarter  count  median_rent  
0  2000Q1  352.0        165.0  
1  2000Q2  347.0        165.0  
2  2000Q3  378.0        170.0  
3  2000Q4  369.0        175.0  
4  2001Q1  395.0        180.0  
areas: 146 | types: 7
quarters: 2000Q1 -> 2025Q3
missing medians: 3.7%


In [5]:
pc = pd.read_csv(RAW / "postcodes.csv")
pc["suburb_u"] = pc.suburb.str.upper().str.strip()
pc = pc[["suburb_u", "postcode"]].drop_duplicates("suburb_u")

xw = area_to_suburbs(panel.area.unique())
xw["suburb_u"] = xw.suburb.str.upper().str.strip()
xw = xw.merge(pc, on="suburb_u", how="left")
xw["suburb_key"] = [make_suburb_key(s, p) if pd.notna(p) else None
                    for s, p in zip(xw.suburb, xw.postcode)]

print(f"{xw.suburb_key.notna().sum()}/{len(xw)} mapped")
print(xw[xw.suburb_key.isna()][["dffh_area", "suburb"]])

210/214 mapped
                           dffh_area           suburb
60   Dandenong North-Endeavour Hills  Dandenong North
86           Flora Hill-Bendigo East       Flora Hill
87           Flora Hill-Bendigo East     Bendigo East
211                     Yarra Ranges     Yarra Ranges


In [7]:
UNMAPPED_NOTES = {
    "Dandenong North": "absent from Domain listing data",
    "Flora Hill":      "Bendigo suburb; absent from Domain listing data",
    "Bendigo East":    "absent from Domain listing data",
    "Yarra Ranges":    "LGA, not a suburb; no single suburb equivalent",
}

In [8]:
from pathlib import Path
Path("../data/curated").mkdir(parents=True, exist_ok=True)
xw.to_csv("../data/curated/dffh_area_suburb_crosswalk.csv", index=False)
panel.to_parquet("../data/curated/dffh_rent_panel.parquet", index=False)
print("saved", xw.shape, panel.shape)

saved (214, 5) (105266, 6)
